# RNAscope Experimenter Counts

Load experimenter counts stored in each NWB file.

In [ ]:
from pathlib import Path
import os
import json
import re
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, to_rgba

import pandas as pd
from IPython.display import display, Markdown
from pynwb import NWBHDF5IO

repo = Path(os.environ.get(
    "ANALYSIS_ROOT",
    Path.home() / "Documents" / "Repositories" / "analysis_Belal2026"
))
functions_dir = repo / "Python functions"

if str(functions_dir) not in sys.path:
    sys.path.insert(0, str(functions_dir))

from master_RNAscope import (
    RNAscopeAnalysisFinish,
    reconstruct_state_from_saved_analysis,
    save_rnascope_field_analysis,
    parse_field_metadata,
    counts_from_roi_jsons,
)

from master_functions import boxplot_rtype

nwb_root = repo / "NWBdata" / "001832"

sessions = ["sub-L1-ST8_ses-20240905T115902", 
            "sub-L2-ST8_ses-20240909T100245", 
            "sub-L3-ST6_ses-20240909T112846", 
            "sub-L4-ST8_ses-20240916T101347"]

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

display(Markdown(f"NWB root: `{nwb_root}`"))

In [ ]:
session_paths = {
    "L1.ST8": {
        "session_root": nwb_root / "sub-L1-ST8",
        "nwb_path": nwb_root / "sub-L1-ST8" / "sub-L1-ST8_ses-20240905T115902.nwb",
    },
    "L2.ST8": {
        "session_root": nwb_root / "sub-L2-ST8",
        "nwb_path": nwb_root / "sub-L2-ST8" / "sub-L2-ST8_ses-20240909T100245.nwb",
    },
    "L3.ST6": {
        "session_root": nwb_root / "sub-L3-ST6",
        "nwb_path": nwb_root / "sub-L3-ST6" / "sub-L3-ST6_ses-20240909T112846.nwb",
    },
    "L4.ST8": {
        "session_root": nwb_root / "sub-L4-ST8",
        "nwb_path": nwb_root / "sub-L4-ST8" / "sub-L4-ST8_ses-20240916T101347.nwb",
    },
}

experimenter_dfs = []

for session, paths in session_paths.items():
    nwb_path = paths["nwb_path"]

    with NWBHDF5IO(str(nwb_path), "r", load_namespaces=True) as io:
        nwbfile = io.read()
        df = (
            nwbfile.processing["rnascope_analysis_metadata"]["experimenter_chrnb2_counts"]
            .to_dataframe()
            .reset_index(drop=True)
        )

    df["session"] = df["session"] if "session" in df.columns else session
    df["session_group"] = df["session_group"] if "session_group" in df.columns else session
    experimenter_dfs.append(df)

experimenter_counts = pd.concat(experimenter_dfs, ignore_index=True)
experimenter_counts["field_index"] = experimenter_counts["field_index"].astype(int)
experimenter_counts["replicate"] = experimenter_counts["replicate"].astype(int)
experimenter_counts["count"] = experimenter_counts["count"].astype(int)

display(experimenter_counts)

In [ ]:
# Optional CSV Export
save_csv = False

if save_csv:
    out_dir = repo / "Paper analysis" / "Figure 11" / "xlsx"
    out_dir.mkdir(parents=True, exist_ok=True)

    counts_csv = out_dir / "RNAscope_experimenter_counts.csv"
    experimenter_counts.to_csv(counts_csv, index=False)


In [ ]:
save_plot = False

SVG_DIR = repo / "Paper analysis" / "Figure 11" / "svg"

cell_type = "NDNF"  # or "TH"

axis_gray = "#666666"

if cell_type == "NDNF":
    cell_marker = "NDNF+"
    ylabel = "CHRNB2 Particles/NDNF Cell"
    yrange = (0, 60)
elif cell_type == "TH":
    cell_marker = "TH+"
    ylabel = "CHRNB2 Particles/TH Cell"
    yrange = (0, 20)
else:
    raise ValueError("cell_type must be 'NDNF' or 'TH'")

df = experimenter_counts.sort_values(
    ["condition", "cell_type", "hemisphere", "slice_id"],
    kind="stable",
).reset_index(drop=True)

df = df[df["cell_type"] == cell_marker].copy()
df["Animal"] = df["slice_id"].astype(str).str.replace(r"^(L\d+).*", r"\1", regex=True)
df["Group"] = np.where(df["hemisphere"].eq("UL"), "Control", "6OHDA")
df["Group"] = pd.Categorical(df["Group"], categories=["Control", "6OHDA"], ordered=True)
df["var"] = df["count"].astype(float)

ctrl = df[df["Group"] == "Control"]
test = df[df["Group"] == "6OHDA"]

viridis_stops = ["#440154", "#3B528B", "#21918C", "#5DC963", "#FDE725"]
field_cmap = LinearSegmentedColormap.from_list("r_boxplot_viridis", viridis_stops)

fields = pd.unique(df["field"])
field_colors = {
    field: to_rgba(field_cmap(i / max(len(fields) - 1, 1)), alpha=0.6)
    for i, field in enumerate(fields)
}

fig, ax = plt.subplots(figsize=(3, 3.5))
fig.patch.set_alpha(0)
ax.set_facecolor("none")

box_plot = boxplot_rtype(
    ax,
    [ctrl["var"].values, test["var"].values],
    rtype=6,
    whis=1.5,
    positions=[1, 2],
    patch_artist=True,
    widths=0.3,
    showfliers=False,
    boxprops=dict(edgecolor=axis_gray, linewidth=4/3),
    whiskerprops=dict(color=axis_gray, linewidth=4/3),
    capprops=dict(color=axis_gray, linewidth=4/3),
    median_overhang=0.03,
    medianprops=dict(color=axis_gray, linewidth=4),
    showpoints=False,
    paired=False,
)

for box in box_plot["boxes"]:
    box.set_facecolor("none")
    box.set_edgecolor(axis_gray)

rng = np.random.default_rng(42)
amount = 0.05
point_size = (0.6 * 10) ** 2

for xpos, group_name in zip([1, 2], ["Control", "6OHDA"]):
    group_df = df[df["Group"] == group_name]

    x = xpos + rng.uniform(-amount, amount, size=len(group_df))
    colors = [field_colors[field] for field in group_df["field"]]

    ax.scatter(
        x,
        group_df["var"],
        s=point_size,
        c=colors,
        edgecolors="none",
        zorder=3,
    )

ax.set_xticks([1, 2])
ax.set_xticklabels(["Control", "6OHDA"], rotation=45, ha="right", color=axis_gray)
ax.set_ylabel(ylabel, color=axis_gray)
ax.set_ylim(*yrange)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_color(axis_gray)
ax.spines["bottom"].set_color(axis_gray)
ax.spines["left"].set_linewidth(4/3)
ax.spines["bottom"].set_linewidth(4/3)

ax.tick_params(
    axis="both",
    direction="out",
    length=4,
    width=4/3,
    colors=axis_gray,
    labelcolor=axis_gray,
)

plt.tight_layout()

plt.rcParams["figure.facecolor"] = "none"
plt.rcParams["axes.facecolor"] = "none"
plt.rcParams["savefig.transparent"] = True

if save_plot:
    SVG_DIR.mkdir(parents=True, exist_ok=True)
    boxplot_svg_path = SVG_DIR / f"Figure 11_RNAscope_boxplot_{cell_type}_PYTHON.svg"
    fig.savefig(boxplot_svg_path, format="svg", bbox_inches="tight", transparent=True)


plt.show()